In [1]:
# import libraries
from dd.autoref import BDD
from itertools import product
import math
import operator

from __future__ import annotations
from typing import Any

import graphviz

import numpy as np
from scipy.stats import norm

In [2]:
# source code
from utils_ADD import *
from utils_transition import *

In [ ]:
#!pip install graphviz
#!sudo apt-get install graphviz

#### Create state and action spaces

In [32]:
# create states and actions
states = [0, 50, 100, 150]
n_states = len(states)
subsystems = ["X", "Y", "Z"]
action_names = ["a1", "a2", "a3"]

# action_set: each column is an action, rows are (a_X, a_Y, a_Z)
# col 0 -> action applied to X only
# col 1 -> action applied to Y only
# col 2 -> action applied to Z only
action_set = np.array([[1, 0, 0],
                       [0, 1, 0],
                       [0, 0, 1]])
action_cost = {"a1": 10, "a2": 10, "a3": 0}

#### Create instance of ADD manager

In [4]:
mgr = ADD(states)

#### Create reward ADD

##### Create reward function

In [5]:
# reward function
def reward_fn(s, a):
    return s["X"] + s["Y"] + s["Z"] + action_cost[a]

##### Build adds

In [6]:
reward_adds = {
    a: mgr.build(lambda s, a=a: reward_fn(s, a), subsystems)
    for a in action_cost
}

##### Test evaluation

In [8]:
print(mgr.evaluate(reward_adds["a1"], {"X": 50, "Y": 100, "Z": 150}))

##### Count nodes in each ADD

In [9]:
for a, add in reward_adds.items():
    print(f"{a}: {mgr.count_nodes(add)} nodes, {mgr.count_leaves(add)} leaves")

##### Visualize ADD for a1

In [51]:
dot_str = mgr.to_dot(reward_adds["a1"], title="Reward ADD (a1)")
graphviz.Source(dot_str)

### Create transition ADD

##### Parameter values for biological dynamics

In [10]:
params = {
    "p_detect": 0.8, # probability of detection
    "st_mean": 30, # mean of shared annual random intercept
    "st_sd": 50, # sd of shared annual random intercept
    "rho": 2, # growth rate
    "dd": 150, # carrying capacity
    "p_increase": 1 
}

##### Function for getting transition matrix for one subsystem and one action


In [11]:
# transition matrix for one subsystem and one action
def get_transition(params, states, a_binary):
    """
    Returns T[s', s] = P(s' | s, a) for a single subsystem and action.
    a_binary: 1 if action applied to this subsystem, 0 otherwise.
    """
    states = np.array(states, dtype=float)
    n = len(states)

    # get probability of capture
    pcap = min(params["p_detect"] * params["p_increase"], 1.0) if a_binary else 0.0

    # random intercept (i.e., environmental variation)
    prob_int = norm.pdf(states, params["st_mean"], params["st_sd"])
    prob_int = prob_int / prob_int.sum()

    out = np.zeros((n, n))

    for j, s in enumerate(states):

        # remaining after capture
        survived = s * (1 - pcap)
        
        lambda_next = (
            states + survived +
            params["rho"] * survived * (1 - survived / params["dd"])
        )

        prob = np.zeros(n)
        for i, lam in enumerate(lambda_next):
            idx = np.argmin(np.abs(lam - states))
            prob[idx] += prob_int[i]

        out[:, j] = prob / prob.sum()

    return out   # shape: (n_next, n_current)

##### Check that transition matrix is different when action is applied vs. not applied

In [27]:
# action applied
T_applied = get_transition(params, states, a_binary=1)
print(T_applied)

In [28]:
# action not applied
T_not_applied = get_transition(params, states, a_binary=0)
print(T_not_applied)

##### Build transition ADDs

In [12]:
# get number of actions
n_actions = action_set.shape[1]

# get transition adds
transition_adds = build_transition_adds(mgr, get_transition, params, states, action_set, n_actions,
                                        subsystems)

In [16]:
# Evaluate: P(X'=100, Y'=50, Z'=0 | X=50, Y=100, Z=150, a=0)
assignment = {"X": 50, "Y": 100, "Z": 150,
              "X_p": 100, "Y_p": 50, "Z_p": 0}
print(mgr.evaluate(transition_adds[0]["Z"], assignment))

0.3814311104263876


##### Show transition ADDs for different actions

In [34]:
# Action 0, subsystem Z
dot_a0 = mgr.to_dot(transition_adds[0]["Z"], title="P(Z'|Z, a=0)")
graphviz.Source(dot_a0)

In [36]:
# Action 1, subsystem Z
dot_a1 = mgr.to_dot(transition_adds[1]["Z"], title="P(Z'|Z, a=1)")
graphviz.Source(dot_a1)

In [37]:
# Action 2, subsystem Z
dot_a2 = mgr.to_dot(transition_adds[2]["Z"], title="P(Z'|Z, a=1)")
graphviz.Source(dot_a2)

##### Compare to transition matrix to show that the transition ADD is equivalent

In [38]:
# compare to transition matrix
T = get_transition(params, states, a_binary=1)
print(T)

## SPUDD algorithm

##### Set discount and tolerance threshold for SPUDD value iteration algorithm

In [ ]:
DISCOUNT = 0.95
TOLERANCE = 0.01

##### Run SPUDD algorithm

In [23]:
V, policy_adds = spudd(
    mgr,
    reward_adds,
    transition_adds,
    discount=DISCOUNT,
    tolerance=TOLERANCE
)

##### Get the optimal action for an example state

In [25]:
get_optimal_action(mgr, policy_adds, {"X": 50, "Y": 100, "Z": 0})

##### Get the optimal value for each state (optimal value function)

In [29]:
df = get_value_df(mgr, states)
print(df)

##### Plot the optimal value for each state (optimal value function)

In [33]:
plt = plot_qvalues(reward_adds, states, action_names)
plt.show()